In [8]:
from mainnet_launch.pages.asset_discounts.stable_coin_pricing import (
    StableCoinConsants,
    stablecoin_constants,
    build_balancer_query_swap_call,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    get_state_by_one_block,
    build_blocks_to_use,
    _build_blocks_to_use_dont_clip,
    safe_normalize_with_bool_success,
    identity_with_bool_success,
)
from mainnet_launch.constants import ETH_CHAIN
import plotly.express as px
import plotly.io as pio
import pandas as pd

from multicall import Call


pio.templates.default = None
import numpy as np

blocks = build_blocks_to_use(ETH_CHAIN, start_block=20689810)

In [9]:
aUSDT = "0x7Bc3485026Ac48b6cf9BaF0A377477Fff5703Af8"
aUSDC = "0xD4fa2D31b7968E448877f69A96DE69f5de8cD23E"
aGHO = "0xC71Ea051a5F82c67ADcF634c36FFE6334793D24C"

aUSDT_backing = Call(
    aUSDT,
    ["convertToAssets(uint256)(uint256)", int(1e18)],
    [("aUSDT_backing", safe_normalize_with_bool_success)],
)

aUSDC_backing = Call(
    aUSDC,
    ["convertToAssets(uint256)(uint256)", int(1e18)],
    [("aUSDC_backing", safe_normalize_with_bool_success)],
)

aGHO_backing = Call(
    aGHO,
    ["convertToAssets(uint256)(uint256)", int(1e18)],
    [("aGHO_backing", safe_normalize_with_bool_success)],
)
# build a convert to Assets Call

aave_backing_calls = [aUSDT_backing, aUSDC_backing, aGHO_backing]
get_state_by_one_block(aave_backing_calls, 22_000_000, ETH_CHAIN)

{'aUSDT_backing': 1.1173073840095442,
 'aUSDC_backing': 1.122906998122201,
 'aGHO_backing': 1.0019473041235327}

In [27]:
balancer_batch_router_address = "0x136f1EFcC3f8f88516B9E94110D56FDBfB1778d1"


def _normalize_6_first_value(success, amountOutList):
    if success:
        return amountOutList[0] /1e6

def make_balancer_router_query(
    name, 
    pool_address,
    token_in,
    token_out,
    amount_in,
):
    # simple one hop path

    paths = [
        (
            token_in,  # tokenIn
            [(pool_address, token_out, False)],  # steps as a list of SwapPathStep tuples
            int(amount_in),  # exactAmountIn as an integer
            0,  # minAmountOut
        )
    ]
    return Call(
        balancer_batch_router_address,
        [
            "querySwapExactIn((address,(address,address,bool)[],uint256,uint256)[],address,bytes)(uint256[],address[],uint256[])",
            paths,
            "0x0000000000000000000000000000000000000000",
            b"",
        ],
        [(name, _normalize_6_first_value),],
    )


Balancer_GHO_USDC_USDT_pool = "0x85B2b559bC2D21104C4DEFdd6EFcA8A20343361D"
aGHO_to_aUSDC_spot_price = make_balancer_router_query('aGHO_to_aUSDC_spot_price', Balancer_GHO_USDC_USDT_pool, aGHO, aUSDC, 1e18)
get_state_by_one_block([c], 22_000_000, ETH_CHAIN)

aave_df = get_raw_state_by_blocks([*aave_backing_calls, aGHO_to_aUSDC_spot_price], blocks, ETH_CHAIN)
aave_df['GHO_to_USDC_spot_price'] = aave_df['aGHO_to_aUSDC_spot_price'].mul(aave_df['aUSDC_backing']).div(aave_df['aGHO_backing'])
px.line(aave_df['GHO_to_USDC_spot_price'])

In [ ]:
break

In [18]:
abi ='''[{"inputs":[{"internalType":"contract IVault","name":"vault","type":"address"},{"internalType":"contract IWETH","name":"weth","type":"address"},{"internalType":"contract IPermit2","name":"permit2","type":"address"},{"internalType":"string","name":"routerVersion","type":"string"}],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"address","name":"target","type":"address"}],"name":"AddressEmptyCode","type":"error"},{"inputs":[{"internalType":"address","name":"account","type":"address"}],"name":"AddressInsufficientBalance","type":"error"},{"inputs":[],"name":"ErrorSelectorNotFound","type":"error"},{"inputs":[],"name":"EthTransfer","type":"error"},{"inputs":[],"name":"FailedInnerCall","type":"error"},{"inputs":[],"name":"InputLengthMismatch","type":"error"},{"inputs":[],"name":"InsufficientEth","type":"error"},{"inputs":[],"name":"ReentrancyGuardReentrantCall","type":"error"},{"inputs":[{"internalType":"uint8","name":"bits","type":"uint8"},{"internalType":"uint256","name":"value","type":"uint256"}],"name":"SafeCastOverflowedUintDowncast","type":"error"},{"inputs":[{"internalType":"address","name":"token","type":"address"}],"name":"SafeERC20FailedOperation","type":"error"},{"inputs":[{"internalType":"address","name":"sender","type":"address"}],"name":"SenderIsNotVault","type":"error"},{"inputs":[],"name":"SwapDeadline","type":"error"},{"inputs":[],"name":"TransientIndexOutOfBounds","type":"error"},{"inputs":[],"name":"getSender","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"bytes[]","name":"data","type":"bytes[]"}],"name":"multicall","outputs":[{"internalType":"bytes[]","name":"results","type":"bytes[]"}],"stateMutability":"payable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"address","name":"owner","type":"address"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"amount","type":"uint256"},{"internalType":"uint256","name":"nonce","type":"uint256"},{"internalType":"uint256","name":"deadline","type":"uint256"}],"internalType":"struct IRouterCommon.PermitApproval[]","name":"permitBatch","type":"tuple[]"},{"internalType":"bytes[]","name":"permitSignatures","type":"bytes[]"},{"components":[{"components":[{"internalType":"address","name":"token","type":"address"},{"internalType":"uint160","name":"amount","type":"uint160"},{"internalType":"uint48","name":"expiration","type":"uint48"},{"internalType":"uint48","name":"nonce","type":"uint48"}],"internalType":"struct IAllowanceTransfer.PermitDetails[]","name":"details","type":"tuple[]"},{"internalType":"address","name":"spender","type":"address"},{"internalType":"uint256","name":"sigDeadline","type":"uint256"}],"internalType":"struct IAllowanceTransfer.PermitBatch","name":"permit2Batch","type":"tuple"},{"internalType":"bytes","name":"permit2Signature","type":"bytes"},{"internalType":"bytes[]","name":"multicallData","type":"bytes[]"}],"name":"permitBatchAndCall","outputs":[{"internalType":"bytes[]","name":"results","type":"bytes[]"}],"stateMutability":"payable","type":"function"},{"inputs":[{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"exactAmountIn","type":"uint256"},{"internalType":"uint256","name":"minAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountIn[]","name":"paths","type":"tuple[]"},{"internalType":"address","name":"sender","type":"address"},{"internalType":"bytes","name":"userData","type":"bytes"}],"name":"querySwapExactIn","outputs":[{"internalType":"uint256[]","name":"pathAmountsOut","type":"uint256[]"},{"internalType":"address[]","name":"tokensOut","type":"address[]"},{"internalType":"uint256[]","name":"amountsOut","type":"uint256[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"sender","type":"address"},{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"exactAmountIn","type":"uint256"},{"internalType":"uint256","name":"minAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountIn[]","name":"paths","type":"tuple[]"},{"internalType":"uint256","name":"deadline","type":"uint256"},{"internalType":"bool","name":"wethIsEth","type":"bool"},{"internalType":"bytes","name":"userData","type":"bytes"}],"internalType":"struct IBatchRouter.SwapExactInHookParams","name":"params","type":"tuple"}],"name":"querySwapExactInHook","outputs":[{"internalType":"uint256[]","name":"pathAmountsOut","type":"uint256[]"},{"internalType":"address[]","name":"tokensOut","type":"address[]"},{"internalType":"uint256[]","name":"amountsOut","type":"uint256[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"maxAmountIn","type":"uint256"},{"internalType":"uint256","name":"exactAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountOut[]","name":"paths","type":"tuple[]"},{"internalType":"address","name":"sender","type":"address"},{"internalType":"bytes","name":"userData","type":"bytes"}],"name":"querySwapExactOut","outputs":[{"internalType":"uint256[]","name":"pathAmountsIn","type":"uint256[]"},{"internalType":"address[]","name":"tokensIn","type":"address[]"},{"internalType":"uint256[]","name":"amountsIn","type":"uint256[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"sender","type":"address"},{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"maxAmountIn","type":"uint256"},{"internalType":"uint256","name":"exactAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountOut[]","name":"paths","type":"tuple[]"},{"internalType":"uint256","name":"deadline","type":"uint256"},{"internalType":"bool","name":"wethIsEth","type":"bool"},{"internalType":"bytes","name":"userData","type":"bytes"}],"internalType":"struct IBatchRouter.SwapExactOutHookParams","name":"params","type":"tuple"}],"name":"querySwapExactOutHook","outputs":[{"internalType":"uint256[]","name":"pathAmountsIn","type":"uint256[]"},{"internalType":"address[]","name":"tokensIn","type":"address[]"},{"internalType":"uint256[]","name":"amountsIn","type":"uint256[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"exactAmountIn","type":"uint256"},{"internalType":"uint256","name":"minAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountIn[]","name":"paths","type":"tuple[]"},{"internalType":"uint256","name":"deadline","type":"uint256"},{"internalType":"bool","name":"wethIsEth","type":"bool"},{"internalType":"bytes","name":"userData","type":"bytes"}],"name":"swapExactIn","outputs":[{"internalType":"uint256[]","name":"pathAmountsOut","type":"uint256[]"},{"internalType":"address[]","name":"tokensOut","type":"address[]"},{"internalType":"uint256[]","name":"amountsOut","type":"uint256[]"}],"stateMutability":"payable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"sender","type":"address"},{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"exactAmountIn","type":"uint256"},{"internalType":"uint256","name":"minAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountIn[]","name":"paths","type":"tuple[]"},{"internalType":"uint256","name":"deadline","type":"uint256"},{"internalType":"bool","name":"wethIsEth","type":"bool"},{"internalType":"bytes","name":"userData","type":"bytes"}],"internalType":"struct IBatchRouter.SwapExactInHookParams","name":"params","type":"tuple"}],"name":"swapExactInHook","outputs":[{"internalType":"uint256[]","name":"pathAmountsOut","type":"uint256[]"},{"internalType":"address[]","name":"tokensOut","type":"address[]"},{"internalType":"uint256[]","name":"amountsOut","type":"uint256[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"maxAmountIn","type":"uint256"},{"internalType":"uint256","name":"exactAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountOut[]","name":"paths","type":"tuple[]"},{"internalType":"uint256","name":"deadline","type":"uint256"},{"internalType":"bool","name":"wethIsEth","type":"bool"},{"internalType":"bytes","name":"userData","type":"bytes"}],"name":"swapExactOut","outputs":[{"internalType":"uint256[]","name":"pathAmountsIn","type":"uint256[]"},{"internalType":"address[]","name":"tokensIn","type":"address[]"},{"internalType":"uint256[]","name":"amountsIn","type":"uint256[]"}],"stateMutability":"payable","type":"function"},{"inputs":[{"components":[{"internalType":"address","name":"sender","type":"address"},{"components":[{"internalType":"contract IERC20","name":"tokenIn","type":"address"},{"components":[{"internalType":"address","name":"pool","type":"address"},{"internalType":"contract IERC20","name":"tokenOut","type":"address"},{"internalType":"bool","name":"isBuffer","type":"bool"}],"internalType":"struct IBatchRouter.SwapPathStep[]","name":"steps","type":"tuple[]"},{"internalType":"uint256","name":"maxAmountIn","type":"uint256"},{"internalType":"uint256","name":"exactAmountOut","type":"uint256"}],"internalType":"struct IBatchRouter.SwapPathExactAmountOut[]","name":"paths","type":"tuple[]"},{"internalType":"uint256","name":"deadline","type":"uint256"},{"internalType":"bool","name":"wethIsEth","type":"bool"},{"internalType":"bytes","name":"userData","type":"bytes"}],"internalType":"struct IBatchRouter.SwapExactOutHookParams","name":"params","type":"tuple"}],"name":"swapExactOutHook","outputs":[{"internalType":"uint256[]","name":"pathAmountsIn","type":"uint256[]"},{"internalType":"address[]","name":"tokensIn","type":"address[]"},{"internalType":"uint256[]","name":"amountsIn","type":"uint256[]"}],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"version","outputs":[{"internalType":"string","name":"","type":"string"}],"stateMutability":"view","type":"function"},{"stateMutability":"payable","type":"receive"}]'''

In [21]:
from mainnet_launch.abis.abi_signature_from_abi_utils import get_function_and_event_signatures_with_returns
import json
get_function_and_event_signatures_with_returns(json.loads(abi))

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 'getSender()(address)',
 'multicall(bytes[])(bytes[])',
 'permitBatchAndCall((address,address,address,uint256,uint256,uint256)[],bytes[],((address,uint160,uint48,uint48)[],address,uint256),bytes,bytes[])(bytes[])',
 'querySwapExactIn((address,(address,address,bool)[],uint256,uint256)[],address,bytes)(uint256[],address[],uint256[])',
 'querySwapExactInHook((address,(address,(address,address,bool)[],uint256,uint256)[],uint256,bool,bytes))(uint256[],address[],uint256[])',
 'querySwapExactOut((address,(address,address,bool)[],uint256,uint256)[],address,bytes)(uint256[],address[],uint256[])',
 'querySwapExactOutHook((address,(address,(address,address,bool)[],uint256,uint256)[],uint256,bool,bytes))(uint256[],address[],uint256[])',
 'swapExactIn((address,(address,address,bool)[],uint256,uint256)[],uint256,bool,bytes)(uint256[],address[],uint256[])',
 'swapExactInHook((address,(address,(address,a